# Homework 5, Problem 4(e)–(g): classifying satellite images with a CNN

This notebook downloads the EuroSAT land-use images (about 94 MB, the first time only), keeps 250 images from each of 4 classes, and trains the CNN from Problem 4(d). Your tasks are marked **TODO**.

**Setup (simplest):** open this notebook in Google Colab (https://colab.research.google.com, *File → Upload notebook*). PyTorch is already installed there, so just run the cells top to bottom with Shift+Enter. Training runs on Colab's standard CPU in about a minute.

**Or on your own computer:** `pip install torch torchvision matplotlib jupyter`, then `jupyter notebook`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

## Load the data

In [ ]:
# Download EuroSAT and keep 250 images from each of 4 classes
CLASSES = ["Forest", "Highway", "Residential", "SeaLake"]
full = torchvision.datasets.EuroSAT(root="data", download=True, transform=transforms.ToTensor())

g = torch.Generator().manual_seed(42)
images, labels = [], []
for new_label, name in enumerate(CLASSES):
    old_label = full.class_to_idx[name]
    rows = [i for i, (_, y) in enumerate(full.samples) if y == old_label]
    for j in torch.randperm(len(rows), generator=g)[:250]:
        images.append(full[rows[j]][0])
        labels.append(new_label)
X = torch.stack(images)   # 1000 images, each 3 x 64 x 64 (red, green, blue)
y = torch.tensor(labels)

# 80/20 train-test split
perm = torch.randperm(len(y), generator=g)
X, y = X[perm], y[perm]
n_train = int(0.8 * len(y))
X_train, y_train = X[:n_train], y[:n_train]
X_test, y_test = X[n_train:], y[n_train:]
print(X_train.shape, X_test.shape)

In [ ]:
# Look at a few training images
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, img, lab in zip(axes.flat, X_train, y_train):
    ax.imshow(img.permute(1, 2, 0))   # channels last for plotting
    ax.set_title(CLASSES[lab], fontsize=9)
    ax.axis("off")
plt.tight_layout()

## The CNN from Problem 4(d)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),    # Conv 1
            nn.ReLU(),
            nn.MaxPool2d(2),                               # Pool 1
            nn.Conv2d(16, 32, kernel_size=3, padding=1),   # Conv 2
            nn.ReLU(),
            nn.MaxPool2d(2),                               # Pool 2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 64),                   # fully connected
            nn.ReLU(),
            nn.Linear(64, 4),                              # output: one score (logit) per class
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("CNN parameters:", count_parameters(CNN()))

## Training loop

In [ ]:
def train(model, epochs=15, batch_size=32, lr=1e-3):
    torch.manual_seed(42)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()   # softmax + cross-entropy, so the model outputs logits
    for epoch in range(epochs):
        model.train()
        for batch in torch.randperm(len(y_train)).split(batch_size):
            optimizer.zero_grad()
            loss = criterion(model(X_train[batch]), y_train[batch])
            loss.backward()
            optimizer.step()
        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(f"epoch {epoch + 1}: last batch loss {loss.item():.3f}")
    return model

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).argmax(dim=1)

In [ ]:
torch.manual_seed(42)
cnn = train(CNN())
pred = predict(cnn, X_test)
print("CNN test accuracy:", (pred == y_test).float().mean().item())

## TODO (part e): a grid of test images with predicted and actual classes, and a confusion matrix

In [ ]:
# TODO

## TODO (part f): a feedforward network on the flattened images

Flatten, one hidden layer of 64 units with ReLU, and 4 outputs. Train it with the same `train` function and compare its parameter count and test accuracy with the CNN.

In [ ]:
# TODO

## TODO (part g): which classes does the CNN confuse? Look at some misclassified test images.

In [ ]:
# TODO